# Retrain the DeepParse mask model (Colab)

Deliberate retrain, run top to bottom on a GPU runtime (T4 is enough):

1. Build the training set: Loghub-2k (50 lines/system), IoT fleet, Loghub-2.0 samples. The `HOLDOUT` system is left out of every source.
2. QLoRA fine-tune DeepSeek-R1-Distill-Llama-8B.
3. Push the adapter to the Hugging Face Hub. A login prompt asks for a **write** token.
4. Write one mask bundle per system with the new adapter and the current one, score both on CPU, and download the bundles.

Every step resumes: data, checkpoints and mask bundles go to `WORK` (Google Drive when `USE_DRIVE`).

In [ ]:
REPO = "https://github.com/Niang931/Log_parser.git"  #@param {type:"string"}
BRANCH = "feat/mask-recalibration"  #@param {type:"string"}
HOLDOUT = "OpenSSH"  #@param {type:"string"}
LOGHUB2_PER_TEMPLATE = 1  #@param {type:"integer"}
EPOCHS = 3  #@param {type:"number"}
HF_REPO = "jakekato/deepparse-finetuned"  #@param {type:"string"}
OLD_ADAPTER = "MThien213/deepparse-iot-r1-8b"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}

In [ ]:
import os
import subprocess

WORK = "/content/drive/MyDrive/logparser/retrain" if USE_DRIVE else "/content/retrain"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)

ROOT = "/content/Log_parser"
if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, ROOT], check=True)
DP = f"{ROOT}/DeepParse"
import sys; sys.path.insert(0, DP)  # editable install is not on sys.path until a restart
CKPT = f"{WORK}/checkpoints/v2-holdout-{HOLDOUT}"
DATA_OUT = f"{WORK}/data"
ZIPS = f"{WORK}/loghub2_zips"
MASKS = f"{WORK}/masks"
!pip install -q -e "{DP}[train]" bitsandbytes huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Training set

In [ ]:
import urllib.request
from pathlib import Path

from deepparse.tools.fetch_loghub import DEFAULT_SYSTEMS

Path(ZIPS).mkdir(parents=True, exist_ok=True)
for s in DEFAULT_SYSTEMS:  # Loghub-2.0, Zenodo record 8275861 (CC-BY-4.0)
    dst = Path(ZIPS) / f"{s}.zip"
    if s == HOLDOUT or dst.exists():
        continue
    url = f"https://zenodo.org/api/records/8275861/files/{s}.zip/content"
    urllib.request.urlretrieve(url, f"{dst}.part")
    os.replace(f"{dst}.part", dst)
    print(s, dst.stat().st_size // 1_000_000, "MB")

%cd {DP}
!python -m deepparse.tools.retrain data --holdout {HOLDOUT} --out {DATA_OUT} \
    --loghub2-zips {ZIPS} --loghub2-per-template {LOGHUB2_PER_TEMPLATE}

## 2. Fine-tune

Resumes from the last checkpoint in `CKPT` if the runtime dropped. Smoke test first? Add `--max-steps 3`.

In [ ]:
%cd {DP}
!python -m deepparse.training.finetune --config configs/train_iot.yaml \
    --train {DATA_OUT}/train.jsonl --output-dir {CKPT} --epochs {EPOCHS} --resume

## 3. Push to the Hub

Paste a token with **write** access when prompted. The repo is created private; make it public on the Hub if you want.

In [ ]:
import shutil

from huggingface_hub import create_repo, login, upload_folder

login()
shutil.copy(f"{DATA_OUT}/manifest.json", f"{CKPT}/training_manifest.json")
create_repo(HF_REPO, private=True, exist_ok=True)
upload_folder(repo_id=HF_REPO, folder_path=CKPT, ignore_patterns=["checkpoint-*", "runs/*"],
              commit_message=f"v2 adapter, holdout {HOLDOUT}, {EPOCHS} epochs")

## 4. Mask bundles + score

One model load per adapter; finished systems are skipped on re-run.

In [ ]:
import json

from deepparse.synth.hf_deepseek_r1 import _load_model, synthesize_hf_from_checkpoint
from deepparse.utils.sampling import deterministic_sample
from huggingface_hub import snapshot_download

systems = DEFAULT_SYSTEMS + ["IoT_fleet"]
for name, adapter in (("v2", CKPT), ("v1", snapshot_download(OLD_ADAPTER))):
    out = Path(MASKS) / name
    out.mkdir(parents=True, exist_ok=True)
    for s in systems:
        if (out / f"{s}.json").exists():
            continue
        lines = [l for l in Path(f"{DP}/artifacts/data/{s}/raw.log").read_text().splitlines() if l.strip()]
        masks = synthesize_hf_from_checkpoint(adapter, deterministic_sample(lines, 50))
        (out / f"{s}.json").write_text(json.dumps([m.to_dict() for m in masks], indent=2))
        print(name, s, len(masks), "masks")
    _load_model.cache_clear()

In [ ]:
%cd {DP}
!python -m deepparse.tools.retrain eval --holdout {HOLDOUT} \
    --masks v2={MASKS}/v2 --masks v1={MASKS}/v1 --out {WORK}/eval

In [ ]:
!cd {WORK} && zip -qr masks.zip masks eval data/manifest.json
from google.colab import files

files.download(f"{WORK}/masks.zip")